~~~
Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
~~~

# 使用Hugging Face快速開始

<table><tbody><tr> <td style="text-align: center">    <a href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/TxGemma/[TxGemma]Quickstart_with_Hugging_Face.ipynb">
      <img alt="Google Colab logo" src="https://www.tensorflow.org/images/colab_logo_32px.png" width="32px"><br> Run in Google Colab
    </a>
</td> <td style="text-align: center">    <a href="https://github.com/google-gemini/gemma-cookbook/blob/main/TxGemma/%5BTxGemma%5DQuickstart_with_Hugging_Face.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> View on GitHub
    </a>
</td> <td style="text-align: center">    <a href="https://huggingface.co/collections/google/txgemma-release-67dd92e931c857d15e4d1e87">
      <img alt="Hugging Face logo" src="https://huggingface.co/front/assets/huggingface_logo-noborder.svg" width="32px"><br> View on Hugging Face
    </a>
</td>
</tr></tbody></table>
此notebook 提供了使用TxGemma 的基本演示，TxGemma 是基於Gemma 2 構建的大型語言模型的集合，可根據治療相關數據生成預測、分類或文本。它包含以下兩者的獨立使用範例：
- 預測任務，需要狹義的 prompting 形式
- 對話式使用（針對TxGemma-Chat 模型變體），更加靈活，包括多輪交互

要了解有關該模型的更多信息，請訪問[此頁面](https://developers.google.com/health-ai-developer-foundations/txgemma)。

## 設定

要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 TxGemma 模型。開始Colab 會話時選擇合適的runtime。
您可以使用 T4 GPU 免費試用 TxGemma 2B 或 9B*：
1. 在 Colab 視窗的右上角，選擇 **▾（其他連接選項）**。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **T4 GPU**。

*要在 T4 GPU 上使用 TxGemma 9B 執行演示，請使用 int8 量化來減少記憶體使用並加快 inference 速度。請注意，尚未評估量化版本的效能。

### 訪問TxGemma

在開始之前，請確保您有權訪問 Hugging Face 上的 TxGemma 模型：
1. 如果您還沒有Hugging Face 帳戶，您可以點選[此處](https://huggingface.co/join) 免費建立帳戶。
2. 前往 [TxGemma 型號頁面](https://huggingface.co/google/txgemma-2b-predict) 並接受使用條件。

### 設定您的 HF token

點選[此處](https://huggingface.co/settings/tokens)產生Hugging Face `read`存取token並將您的存取token新增至ColabSecrets manager 以安全地儲存它。
1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. 建立一個新的secret，名稱為`HF_TOKEN`。
3. 將token 金鑰複製/貼上到`HF_TOKEN` 的值輸入框中。
4. 切換左側的按鈕以允許notebook 存取secret。

In [ ]:
import os
from google.colab import userdata
# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### 安裝依賴項

In [ ]:
! pip install --upgrade --quiet accelerate bitsandbytes huggingface_hub transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

## 從Hugging Face Hub載入模型

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_VARIANT = "9b-chat"  # @param ["2b-predict", "9b-chat", "9b-predict", "27b-chat", "27b-predict"]

model_id = f"google/txgemma-{MODEL_VARIANT}"

if MODEL_VARIANT == "2b-predict":
    additional_args = {}
else:
    additional_args = {
        "quantization_config": BitsAndBytesConfig(load_in_8bit=True)
    }

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    **additional_args,
)

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

加載後，您可以直接使用模型和tokenizer，這使您可以完全控制inference過程，包括token輸出的化和後處理。
或者，您可以使用 [`pipeline`](https://www.google.com/url?q=https%3A%2F%2Fhuggingface.co%2Fdocs%2Ftransformers%2Fen%2Fmain_classes%2Fpipelines) API，它提供了一種使用inference 模型的簡單方法，同時抽像出複雜的細節。在這裡，使用載入的模型實例化文字產生管道：

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Device set to use cuda:0


以下部分包括示範如何直接使用模型以及與 `pipeline` API 一起使用模型的獨立範例。在實踐中，您應該選擇最適合您的用例的方法。

## Format prompts for therapeutic tasks

本notebook 中的以下部分演示了prompting TxGemma 用於來自[治療數據共享](https://tdcommons.ai/) (TDC) 的治療開發任務。
對於這些預測任務，prompts 應根據 TDC 結構進行格式化，包括：
- **說明：** 簡要描述任務。

- **上下文：** 提供 2-3 個相關生化背景的句子，源自 TDC 描述和文獻。

- **問題：** 查詢特定的治療屬性，將治療和/或目標的文字表示作為輸入。

  - 輸入可以包括 SMILES 字串、胺基酸序列、核苷酸序列和自然語言文字。

  - 可以提供可選的少量範例。

### 載入prompt模板

首先，載入包含各種 TDC 任務的 prompt 格式的 JSON 檔案。

In [ ]:
import json
from huggingface_hub import hf_hub_download

tdc_prompts_filepath = hf_hub_download(
    repo_id=model_id,
    filename="tdc_prompts.json",
)

with open(tdc_prompts_filepath, "r") as f:
    tdc_prompts_json = json.load(f)

tdc_prompts.json:   0%|          | 0.00/768k [00:00<?, ?B/s]

### 準備樣品prompt

使用模板和來自 [BBB（血腦屏障），Martins 等人](https://tdcommons.ai/single_pred_tasks/adme#bbb-blood-brain-barrier-martins-et-al) dataset 的輸入藥物 SMILES 字符串構建 prompt。該 prompt 將用於在下一節中產生預測。
**注意：** 除了替換輸入之外，不應從模板中修改prompt。

In [ ]:
# Set example task and input
task_name = "BBB_Martins"
input_type = "{Drug SMILES}"
drug_smiles = "CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21"

TDC_PROMPT = tdc_prompts_json[task_name].replace(input_type, drug_smiles)
print("Formatted prompt:\n")
print(TDC_PROMPT)

Formatted prompt:

Instructions: Answer the following question about drug properties.
Context: As a membrane separating circulating blood and brain extracellular fluid, the blood-brain barrier (BBB) is the protection layer that blocks most foreign drugs. Thus the ability of a drug to penetrate the barrier to deliver to the site of action forms a crucial challenge in development of drugs for central nervous system.
Question: Given a drug SMILES string, predict whether it
(A) does not cross the BBB (B) crosses the BBB
Drug SMILES: CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21
Answer:


## 探索預測能力

TxGemma 模型旨在處理和理解與各種治療方式和標靶相關的訊息，包括小分子、蛋白質、核酸、疾病和 cell 系，並且可以對廣泛的治療開發任務進行預測。

### 在治療任務上執行inference

本節示範 prompting TxGemma 來自 TDC 的預測任務。

**直接執行模型**

In [ ]:
# Use sample prompt for a predictive task from TDC
prompt = TDC_PROMPT

# Prepare tokenized inputs
input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate response
outputs = model.generate(**input_ids, max_new_tokens=8)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Instructions: Answer the following question about drug properties.
Context: As a membrane separating circulating blood and brain extracellular fluid, the blood-brain barrier (BBB) is the protection layer that blocks most foreign drugs. Thus the ability of a drug to penetrate the barrier to deliver to the site of action forms a crucial challenge in development of drugs for central nervous system.
Question: Given a drug SMILES string, predict whether it
(A) does not cross the BBB (B) crosses the BBB
Drug SMILES: CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21
Answer: (B)


**使用 `pipeline` API 執行**

In [ ]:
# Use sample prompt for a predictive task from TDC
prompt = TDC_PROMPT

# Generate response
outputs = pipe(prompt, max_new_tokens=8)
response = outputs[0]["generated_text"]
print(response)

Instructions: Answer the following question about drug properties.
Context: As a membrane separating circulating blood and brain extracellular fluid, the blood-brain barrier (BBB) is the protection layer that blocks most foreign drugs. Thus the ability of a drug to penetrate the barrier to deliver to the site of action forms a crucial challenge in development of drugs for central nervous system.
Question: Given a drug SMILES string, predict whether it
(A) does not cross the BBB (B) crosses the BBB
Drug SMILES: CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21
Answer: (B)


## 使用 TxGemma-Chat 探索對話功能

TxGemma 具有會話模型，可為預測添加推論和可解釋性，並可用於多輪交互作用。他們的對話能力是以犧牲一些預測能力為代價的。
**對於本部分，請確保您已選擇 TxGemma-Chat 模型變體。 **

### 在多輪對話中提問

本節示範 prompting TxGemma 進行對話使用，遵循 [Gemma 聊天範本](https://ai.google.dev/gemma/docs/core/prompt-structure)。
在此範例中，首先 prompt 模型來回答有關使用 TDC 格式的預測任務的問題。然後，提出後續問題，要求模型提供預測答案的推論。

**直接執行模型**

In [ ]:
from IPython.display import display, Markdown

questions = [
    TDC_PROMPT,  # Initial question is a predictive task from TDC
    "Explain your reasoning based on the molecule structure."
]

messages = []

display(Markdown("\n\n---\n\n"))
for question in questions:
    display(Markdown(f"**User:**\n\n{question}\n\n---\n\n"))
    messages.append(
        { "role": "user", "content": question },
    )
    # Apply the tokenizer's built-in chat template
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    outputs = model.generate(input_ids=inputs.to("cuda"), max_new_tokens=512)
    response = tokenizer.decode(outputs[0, len(inputs[0]):], skip_special_tokens=True)
    display(Markdown(f"**TxGemma:**\n\n{response}\n\n---\n\n"))
    messages.append(
        { "role": "assistant", "content": response},
    )



---



**User:**

Instructions: Answer the following question about drug properties.
Context: As a membrane separating circulating blood and brain extracellular fluid, the blood-brain barrier (BBB) is the protection layer that blocks most foreign drugs. Thus the ability of a drug to penetrate the barrier to deliver to the site of action forms a crucial challenge in development of drugs for central nervous system.
Question: Given a drug SMILES string, predict whether it
(A) does not cross the BBB (B) crosses the BBB
Drug SMILES: CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21
Answer:

---



**TxGemma:**

(B)

---



**User:**

Explain your reasoning based on the molecule structure.

---



**TxGemma:**

Here's the breakdown of why the drug with SMILES CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21 likely crosses the BBB:

* **Small Size:** The molecule is relatively small, which is a general favorable characteristic for BBB penetration. Larger molecules have a harder time squeezing through the tight junctions between brain endothelial cells.
* **Lipophilicity (Hydrophobicity):** The presence of multiple carbon and hydrogen atoms in the benzene rings and alkyl chains makes the molecule predominantly lipophilic (fat-loving).  Lipophilicity is a key determinant of BBB permeability. The more lipophilic a drug, the easier it crosses the lipid-rich cell membranes of the BBB.
* **Lack of Charged Groups:**  The molecule lacks large, charged groups (like carboxyl or amine groups).  Charged groups tend to be repelled by the lipid bilayer of the BBB, making it harder to cross. 
* **Absence of Specific BBB Targets:** While some drugs have specific transporters that help them across the BBB, this molecule doesn't appear to have any obvious targeting groups.

**In summary, the drug's small size, lipophilic nature, lack of significant charge, and absence of specific BBB-targeting groups suggest that it likely possesses a good ability to cross the blood-brain barrier.**

**Important Note:** This is a general prediction based on structural features. Actual BBB permeability is complex and can be influenced by many factors. Experimental validation is always necessary to confirm drug penetration ability. 


---



**使用 `pipeline` API 執行**

In [ ]:
from IPython.display import display, Markdown

questions = [
    TDC_PROMPT,  # Initial question is a predictive task from TDC
    "Explain your reasoning based on the molecule structure."
]

messages = []

display(Markdown("\n\n---\n\n"))
for question in questions:
    display(Markdown(f"**User:**\n\n{question}\n\n---\n\n"))
    messages.append(
        { "role": "user", "content": question },
    )
    outputs = pipe(messages, max_new_tokens=512)
    messages = outputs[0]["generated_text"]
    response = messages[-1]["content"].strip()
    display(Markdown(f"**TxGemma:**\n\n{response}\n\n---\n\n"))



---



**User:**

Instructions: Answer the following question about drug properties.
Context: As a membrane separating circulating blood and brain extracellular fluid, the blood-brain barrier (BBB) is the protection layer that blocks most foreign drugs. Thus the ability of a drug to penetrate the barrier to deliver to the site of action forms a crucial challenge in development of drugs for central nervous system.
Question: Given a drug SMILES string, predict whether it
(A) does not cross the BBB (B) crosses the BBB
Drug SMILES: CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21
Answer:

---



**TxGemma:**

(B)

---



**User:**

Explain your reasoning based on the molecule structure.

---



**TxGemma:**

Here's the breakdown of why the drug with SMILES CN1C(=O)CN=C(C2=CCCCC2)c2cc(Cl)ccc21 likely crosses the BBB:

* **Small Size:** The molecule is relatively small, which is a general favorable characteristic for BBB penetration. Larger molecules have a harder time squeezing through the tight junctions between brain endothelial cells.
* **Lipophilicity (Hydrophobicity):** The presence of multiple carbon and hydrogen atoms in the benzene rings and alkyl chains makes the molecule predominantly lipophilic (fat-loving).  Lipophilicity is a key determinant of BBB permeability. The more lipophilic a drug, the easier it crosses the lipid-rich cell membranes of the BBB.
* **Lack of Charged Groups:**  The molecule lacks large, charged groups (like carboxyl or amine groups).  Charged groups tend to be repelled by the lipid bilayer of the BBB, making it harder to cross. 
* **Absence of Specific BBB Targets:** While some drugs have specific transporters that help them across the BBB, this molecule doesn't appear to have any obvious targeting groups.

**In summary, the drug's small size, lipophilic nature, lack of significant charge, and absence of specific BBB-targeting groups suggest that it likely possesses a good ability to cross the blood-brain barrier.**

**Important Note:** This is a general prediction based on structural features. Actual BBB permeability is complex and can be influenced by many factors. Experimental validation is always necessary to confirm drug penetration ability.

---



# 後續步驟

探索其他 [notebooks](https://github.com/google-gemini/gemma-cookbook/blob/main/TxGemma) 以了解您還可以使用該模型做什麼。